# Experimentos: optimización de hiperparámetros con Optuna

Objetivo: usar Optuna para buscar los mejores hiperparámetros de XGBoost,
rastrear cada trial como un run anidado en MLflow, entrenar el modelo final
con los mejores parámetros y registrarlo en MLflow Model Registry.

In [ ]:
from __future__ import annotations

import warnings

import mlflow
import pandas as pd

from healthcare_fraud.data import clean_dataframe, load_dataset, validate_dataframe
from healthcare_fraud.features import (
    FEATURE_COLS,
    build_features,
    prepare_train_val,
    split_providers,
)
from healthcare_fraud.models import (
    optimize_hyperparameters,
    register_model,
    setup_mlflow,
    train_model,
    transition_model_stage,
)

warnings.filterwarnings("ignore")
print("Imports OK")

## 1. Preparación de datos

In [ ]:
raw_tables = load_dataset()
clean_tables = {
    name: clean_dataframe(validate_dataframe(df, name), name) for name, df in raw_tables.items()
}
print("Tablas limpias:", {k: v.shape for k, v in clean_tables.items()})

In [ ]:
feature_df = build_features(clean_tables)
train_df, val_df = split_providers(feature_df)

X_train, X_val, y_train, y_val, preprocessor = prepare_train_val(train_df, val_df)
X_train_raw = train_df[FEATURE_COLS].values
X_val_raw = val_df[FEATURE_COLS].values

n_fraud = int(y_train.sum())
n_total = len(y_train)
print(f"Train: {n_total} providers | Fraude: {n_fraud} ({n_fraud / n_total:.1%})")

## 2. Optimización de hiperparámetros con Optuna

In [ ]:
setup_mlflow()

with mlflow.start_run(run_name="optuna_experiment") as parent_run:
    mlflow.set_tag("notebook", "03_experiments")
    best_params = optimize_hyperparameters(X_train, y_train, X_val, y_val)

print("Mejores hiperparámetros:")
pd.Series(best_params)

## 3. Entrenamiento del modelo final

In [ ]:
with mlflow.start_run(run_name="final_experiment") as parent_run:
    mlflow.set_tag("notebook", "03_experiments")
    run_id, metrics = train_model(X_train_raw, y_train, X_val_raw, y_val, preprocessor, best_params)

print(f"Run ID: {run_id}")
pd.DataFrame([metrics], index=["optimizado"]).round(4)

## 4. Registro en MLflow Model Registry

In [ ]:
MODEL_NAME = "healthcare-fraud-detector"

mv = register_model(run_id, MODEL_NAME)
print(f"Modelo registrado: {mv.name} v{mv.version}")

In [ ]:
mv_staging = transition_model_stage(MODEL_NAME, mv.version, "Staging")
print(f"Stage: {mv_staging.current_stage}")

## Conclusiones

- Los runs de Optuna quedan registrados como runs anidados en MLflow UI.
- Para ver los resultados: `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db`
- El modelo optimizado está en MLflow Registry en stage **Staging**.
- El siguiente paso (Fase 03) es orquestar este flujo completo con Prefect
  y exponer el modelo vía FastAPI en la Fase 04.